<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/04_WOA_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۴: پیاده‌سازی الگوریتم نهنگ باینری (Binary WOA)

در این نوت‌بوک الگوریتم بهینه‌سازی نهنگ (Whale Optimization Algorithm) را قدم‌به‌قدم می‌سازیم و برای **انتخاب ویژگی** به کار می‌بریم.

**ایده‌ی کلی:**
- هر «نهنگ» یک راه‌حل پیشنهادی است. موقعیت نهنگ یک بردار پیوسته با طول برابر تعداد ویژگی‌هاست (هر مؤلفه بین ‎−۶ و ‎+۶).
- موقعیت پیوسته با یک **تابع انتقال سیگموید** به بردار صفر و یک تبدیل می‌شود: ۱ یعنی ویژگی انتخاب شده و ۰ یعنی نشده.
- هر بردار باینری با **تابع fitness** امتیاز می‌گیرد (کمتر = بهتر). fitness ترکیبی از خطای مدل (با CV) و تعداد ویژگی‌های انتخاب‌شده است.
- نهنگ‌ها طبق سه رفتار الگوریتم (محاصره‌ی شکار، جست‌وجوی تصادفی، حرکت مارپیچ) به سمت بهترین راه‌حل می‌روند و این روند چند تکرار ادامه دارد.

**فرمول‌های WOA** (برای هر نهنگ در هر تکرار؛ $t$ شماره‌ی تکرار و $T$ تعداد کل تکرارها):

- ضریب کاهشی: $a = 2 - t \cdot \frac{2}{T}$ (از ۲ به سمت ۰ کم می‌شود)
- $A = 2a\,r_1 - a$ و $C = 2\,r_2$ که $r_1, r_2$ اعداد تصادفی در $[0,1]$ هستند
- $p$ یک عدد تصادفی در $[0,1]$ است که رفتار را انتخاب می‌کند:
  - اگر $p<0.5$ و $|A|<1$: **محاصره** (بهره‌برداری): $D=|C\,X_{best}-X|$ و $X_{new}=X_{best}-A\,D$
  - اگر $p<0.5$ و $|A|\ge 1$: **جست‌وجو** (اکتشاف) با یک نهنگ تصادفی $X_{rand}$: $D=|C\,X_{rand}-X|$ و $X_{new}=X_{rand}-A\,D$
  - اگر $p\ge 0.5$: **مارپیچ**: $D'=|X_{best}-X|$ و $X_{new}=D'\,e^{bl}\cos(2\pi l)+X_{best}$ با $b=1$ و $l\in[-1,1]$

در پایان نتیجه‌ی الگوریتم یک بردار صفر و یک است که ویژگی‌های انتخاب‌شده را مشخص می‌کند. فقط از داده‌ی **train** استفاده می‌شود و test در هیچ مرحله‌ی انتخاب دخالت ندارد.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive وصل می‌شود تا به پوشه‌ی پروژه، دیتاست‌ها و `dataset_config.py` دسترسی داشته باشیم.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<div dir="rtl" style="text-align: right">

### وارد کردن کتابخانه‌ها
- `numpy`: بردارها و عملیات عددی الگوریتم نهنگ و تولید اعداد تصادفی،
- `pandas`: جدول داده و ذخیره‌ی نتایج،
- `matplotlib`: نمودار همگرایی،
- `scikit-learn`: `Pipeline` و `StandardScaler` و `LogisticRegression` برای مدل ارزیاب، و `cross_val_score` برای اعتبارسنجی متقابل در تابع fitness.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully.")

<div dir="rtl" style="text-align: right">

### بارگذاری دیتاست فعال
مسیر پروژه تعریف و تنظیمات دیتاست (`dataset_config.py`) وارد می‌شود. `load_dataset` دیتاست فعال را می‌خواند و ابعاد و چند ردیف اول چاپ می‌شود.

</div>

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

df = load_dataset(PROJECT_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print(df.head())

<div dir="rtl" style="text-align: right">

### آماده‌سازی داده‌ی train و test
- `split_xy` ویژگی‌ها و برچسب و (برای Parkinson) شناسه‌ی افراد را جدا می‌کند.
- `split_train_test` داده را ۸۰٪ / ۲۰٪ تقسیم می‌کند.
- ویژگی‌های تکراری و ثابت فقط از روی train شناسایی و از هر دو مجموعه حذف می‌شوند.

تعداد ویژگی‌های باقی‌مانده طول بردار هر نهنگ را مشخص می‌کند. الگوریتم نهنگ فقط با `X_train` و `y_train` کار خواهد کرد.

</div>

In [ ]:
X, y, groups = split_xy(df)

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

# Remove duplicate features based only on training data
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Number of duplicate features removed:", len(duplicate_columns))
print("Number of available features:", X_train.shape[1])

<div dir="rtl" style="text-align: right">

### تنظیم پارامترهای الگوریتم
- `n_features`: تعداد ویژگی‌ها = طول هر موقعیت.
- `population_size = 10`: تعداد نهنگ‌ها.
- `max_iterations = 20`: تعداد تکرارها.
- `rng`: مولد اعداد تصادفی با seed ثابت ۴۲ تا نتایج تکرارپذیر باشند.

تعداد کل ارزیابی fitness حدود $10 \times (20+1) = 210$ است و هر ارزیابی خودش ۵ بار مدل را آموزش می‌دهد، پس زمان اجرا عمدتاً از همین‌جاست.

</div>

In [ ]:
# Number of available features
n_features = X_train.shape[1]

# WOA parameters
population_size = 10
max_iterations = 20

# Reproducibility
random_state = 42
rng = np.random.default_rng(random_state)

print("Number of features:", n_features)
print("Population size:", population_size)
print("Maximum iterations:", max_iterations)

<div dir="rtl" style="text-align: right">

### مقداردهی اولیه‌ی جمعیت (موقعیت نهنگ‌ها)
هر نهنگ یک بردار با `n_features` مؤلفه است و هر مؤلفه از توزیع یکنواخت روی بازه‌ی $[-6, 6]$ انتخاب می‌شود. ماتریس `positions` شکل `(تعداد نهنگ‌ها, تعداد ویژگی‌ها)` دارد. چون سیگموید در صفر برابر ۰٫۵ است، در شروع تقریباً نیمی از ویژگی‌ها برای هر نهنگ انتخاب می‌شوند. کمینه و بیشینه‌ی چاپ‌شده نشان می‌دهد مقادیر در بازه‌اند.

</div>

In [ ]:
# Initialize whale positions
lower_bound = -6
upper_bound = 6

positions = rng.uniform(
    lower_bound,
    upper_bound,
    size=(population_size, n_features)
)

print("Population shape:", positions.shape)
print("Minimum position:", positions.min())
print("Maximum position:", positions.max())

<div dir="rtl" style="text-align: right">

### نگاهی به موقعیت دو نهنگ
ده مؤلفه‌ی اول موقعیت نهنگ اول و دوم نمایش داده می‌شود تا ببینیم مقادیر پیوسته و تصادفی بین ‎−۶ و ‎+۶ هستند. این فقط برای مشاهده است.

</div>

In [ ]:
print("First whale position (first 10 features):")
print(positions[0, :10])

print("\nSecond whale position (first 10 features):")
print(positions[1, :10])

<div dir="rtl" style="text-align: right">

### تابع محاسبه‌ی ضرایب $a$، $A$ و $C$
- $a = 2 - t\cdot\frac{2}{T}$ با گذشت تکرارها خطی از ۲ به سمت ۰ کم می‌شود.
- $A = 2a\,r_1 - a$ و $C = 2\,r_2$ با دو عدد تصادفی.

$a$ بزرگ (ابتدای اجرا) یعنی $|A|$ می‌تواند بزرگ باشد و نهنگ‌ها بیشتر **جست‌وجو** می‌کنند. با کوچک شدن $a$ مقدار $|A|$ کمتر از ۱ می‌شود و نهنگ‌ها بیشتر به بهترین جواب **نزدیک** می‌شوند. این تابع برای نمایش مفهوم است؛ تابع به‌روزرسانی موقعیت‌ها همین محاسبه را خودش دوباره انجام می‌دهد.

</div>

In [ ]:
def calculate_coefficients(iteration, max_iterations, rng):
    a = 2 - iteration * (2 / max_iterations)

    r1 = rng.random()
    r2 = rng.random()

    A = 2 * a * r1 - a
    C = 2 * r2

    return A, C

<div dir="rtl" style="text-align: right">

### آزمایش تابع ضرایب
تابع با `iteration=0` (ابتدای اجرا، $a=2$) صدا زده می‌شود و $A$ و $C$ چاپ می‌شوند. مقدار $A$ در $[-2, 2]$ و $C$ در $[0, 2]$ است.

</div>

In [ ]:
A, C = calculate_coefficients(
    iteration=0,
    max_iterations=max_iterations,
    rng=rng
)

print("A:", A)
print("C:", C)

<div dir="rtl" style="text-align: right">

### تابع به‌روزرسانی موقعیت نهنگ‌ها (قلب الگوریتم)
برای هر نهنگ $X$:
1. ضرایب $a$، $A$، $C$ و عدد تصادفی $p$ محاسبه می‌شوند.
2. بسته به $p$ و $|A|$ یکی از سه رفتار اجرا می‌شود:
   - **محاصره** ($p<0.5$ و $|A|<1$): نهنگ به سمت بهترین موقعیت می‌رود: `best_position - A * D` با $D=|C\,X_{best}-X|$. این **بهره‌برداری** (exploitation) است.
   - **جست‌وجو** ($p<0.5$ و $|A|\ge1$): یک نهنگ تصادفی از جمعیت انتخاب و نهنگ نسبت به او حرکت می‌کند. این **اکتشاف** (exploration) است و از گیر کردن در یک جواب جلوگیری می‌کند.
   - **مارپیچ** ($p\ge0.5$): نهنگ در یک مسیر مارپیچی (با `exp` و `cos`) دور بهترین موقعیت می‌چرخد، مثل شکار حباب‌ساز نهنگ‌ها.
3. `np.clip` موقعیت جدید را به بازه‌ی $[-6, 6]$ برمی‌گرداند تا مقادیر از حد نگذرند.

`r1`، `r2`، `p` و `l` برای هر نهنگ یک عدد (اسکالر) هستند. خروجی ماتریس موقعیت‌های جدید است.

</div>

In [ ]:
def update_whale_positions(
    positions,
    best_position,
    iteration,
    max_iterations,
    rng,
    lower_bound=-6,
    upper_bound=6
):
    new_positions = np.zeros_like(positions)

    a = 2 - iteration * (2 / max_iterations)

    n_whales = positions.shape[0]

    for i in range(n_whales):

        X = positions[i].copy()

        r1 = rng.random()
        r2 = rng.random()

        A = 2 * a * r1 - a
        C = 2 * r2

        p = rng.random()

        if p < 0.5:

            if abs(A) < 1:

                # Encircling the best solution
                D = np.abs(C * best_position - X)
                new_position = best_position - A * D

            else:

                # Search for a random whale
                random_index = rng.integers(0, n_whales)
                random_position = positions[random_index]

                D = np.abs(C * random_position - X)
                new_position = random_position - A * D

        else:

            # Spiral movement
            b = 1
            l = rng.uniform(-1, 1)

            D_prime = np.abs(best_position - X)

            new_position = (
                D_prime * np.exp(b * l) * np.cos(2 * np.pi * l)
                + best_position
            )

        # Keep positions within the search boundaries
        new_positions[i] = np.clip(
            new_position,
            lower_bound,
            upper_bound
        )

    return new_positions

<div dir="rtl" style="text-align: right">

فعلاً چون هنوز تابع فیتنس واقعی را تعریف نکرده‌ایم، نهنگ اول را به‌صورت آزمایشی به‌عنوان بهترین موقعیت در نظر می‌گیریم. این فقط یک آزمایش فنی است

</div>

<div dir="rtl" style="text-align: right">

### آزمایش تابع به‌روزرسانی
نهنگ اول را به‌صورت آزمایشی «بهترین» می‌گیریم و یک بار موقعیت‌ها را به‌روز می‌کنیم. شکل خروجی باید با ورودی برابر باشد و همه‌ی مقادیر باید در بازه‌ی ‎−۶ تا ‎+۶ بمانند (`All positions within bounds: True`).

</div>

In [ ]:
test_best_position = positions[0].copy()

updated_positions = update_whale_positions(
    positions=positions,
    best_position=test_best_position,
    iteration=0,
    max_iterations=max_iterations,
    rng=rng
)

print("Updated population shape:", updated_positions.shape)
print("Minimum updated position:", updated_positions.min())
print("Maximum updated position:", updated_positions.max())

print(
    "All positions within bounds:",
    np.all(updated_positions >= lower_bound)
    and np.all(updated_positions <= upper_bound)
)

<div dir="rtl" style="text-align: right">

### تبدیل موقعیت پیوسته به باینری (تابع انتقال سیگموید)
- `sigmoid(x) = 1 / (1 + e^{-x})` هر مؤلفه را به احتمال بین ۰ و ۱ تبدیل می‌کند. `np.clip(x, -50, 50)` از سرریز عددی جلوگیری می‌کند.
- در `continuous_to_binary` برای هر ویژگی یک عدد تصادفی در $[0,1]$ گرفته می‌شود و اگر از احتمال کوچک‌تر بود ویژگی انتخاب می‌شود (۱) وگرنه نه (۰).

به این روش تابع انتقال از نوع **S-شکل** و نمونه‌گیری تصادفی می‌گویند: مؤلفه‌ی بزرگ‌تر یعنی احتمال انتخاب بیشتر. چون مؤلفه‌ها بین ‎−۶ و ‎+۶ هستند، احتمال انتخاب بین حدود ۰٫۰۰۲۵ و ۰٫۹۹۷۵ می‌ماند و هیچ ویژگی هرگز کاملاً قطعی نیست.

</div>

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))


def continuous_to_binary(position, rng):
    probabilities = sigmoid(position)

    random_values = rng.random(size=position.shape)

    binary_position = (random_values < probabilities).astype(int)

    return binary_position

<div dir="rtl" style="text-align: right">

### آزمایش تبدیل باینری
موقعیت نهنگ اول به باینری تبدیل می‌شود. طول بردار، چند مقدار اول، و تعداد ویژگی‌های انتخاب‌شده و نشده چاپ می‌شود. در شروع انتظار داریم حدود نیمی از ویژگی‌ها انتخاب شوند.

</div>

In [ ]:
test_binary = continuous_to_binary(
    positions[0],
    rng
)

print("Binary vector shape:", test_binary.shape)
print("First 20 values:", test_binary[:20])
print("Number of selected features:", test_binary.sum())
print("Number of unselected features:", len(test_binary) - test_binary.sum())

<div dir="rtl" style="text-align: right">

### تست‌های درستی (assert)
بررسی می‌کنیم بردار باینری طول درست دارد، فقط شامل ۰ و ۱ است و تعداد انتخاب‌شده‌ها منفی نیست. اگر شرطی برقرار نباشد، برنامه با خطا متوقف می‌شود.

</div>

In [ ]:
assert test_binary.shape == (n_features,)
assert set(np.unique(test_binary)).issubset({0, 1})
assert test_binary.sum() >= 0

print("Binary conversion test passed.")

<div dir="rtl" style="text-align: right">

### تعریف مدل ارزیاب، CV و ضریب $\alpha$
- `classifier`: یک `Pipeline` شامل استانداردسازی و Logistic Regression. این مدل سریع است و در هر ارزیابی fitness (صدها بار) استفاده می‌شود. `class_weight="balanced"` برای داده‌ی نامتوازن است.
- `cv = make_cv(groups_train)`: اعتبارسنجی ۵-تایی روی train (برای Parkinson بر اساس فرد).
- `alpha = 0.9`: وزن خطای مدل در fitness. تعداد ویژگی وزن $1-\alpha = 0.1$ دارد، یعنی دقت مهم‌تر از کوچک بودن مدل است.

</div>

In [ ]:
# Define the classification model
classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

# Define cross-validation
cv = make_cv(groups_train)

# Fitness parameters
alpha = 0.9

print("Classifier and cross-validation are ready.")
print("Alpha:", alpha)
print("Number of CV folds:", cv.get_n_splits())

<div dir="rtl" style="text-align: right">

### تابع fitness
fitness هر راه‌حل باینری را ارزیابی می‌کند (**کمتر بهتر است**):

$$fitness = \alpha\,(1 - F1_{macro}) + (1-\alpha)\,\frac{k}{n}$$

که $k$ تعداد ویژگی‌های انتخاب‌شده و $n$ تعداد کل ویژگی‌هاست.
1. اندیس ویژگی‌های انتخاب‌شده (مقدار ۱) پیدا می‌شود.
2. اگر هیچ ویژگی‌ای انتخاب نشده باشد fitness برابر ۱٫۰ (بدترین حالت) برمی‌گردد.
3. فقط ستون‌های انتخاب‌شده از `X_train` برداشته می‌شوند.
4. `cross_val_score` با CV ۵-تایی میانگین F1 Macro را روی train حساب می‌کند. test استفاده نمی‌شود.
5. fitness از ترکیب خطا ($1-F1$) و نسبت ویژگی‌ها ساخته می‌شود.

بنابراین الگوریتم مدلی با F1 بالا و ویژگی کم را ترجیح می‌دهد.

</div>

In [ ]:
def fitness_function(
    binary_position,
    X_train,
    y_train,
    classifier,
    cv,
    alpha=0.9
):
    # Convert input to a NumPy array
    binary_position = np.asarray(binary_position)

    # Find selected feature indices
    selected_indices = np.where(binary_position == 1)[0]

    # Count selected features
    n_selected = len(selected_indices)

    # Reject solutions with no selected features
    if n_selected == 0:
        return 1.0

    # Select only the chosen features
    X_selected = X_train.iloc[:, selected_indices]

    # Evaluate classification performance using cross-validation
    cv_scores = cross_val_score(
        classifier,
        X_selected,
        y_train,
        cv=cv,
        scoring="f1_macro"
    )

    mean_f1 = cv_scores.mean()

    # Calculate feature selection ratio
    feature_ratio = n_selected / X_train.shape[1]

    # Calculate final fitness
    fitness = (
        alpha * (1 - mean_f1)
        + (1 - alpha) * feature_ratio
    )

    return fitness

<div dir="rtl" style="text-align: right">

### تست fitness: همه‌ی ویژگی‌ها
همه‌ی ویژگی‌ها انتخاب می‌شوند. این fitness «مدل کامل بدون انتخاب ویژگی» است و معیار مرجع می‌شود. چون $k/n=1$، جمله‌ی تعداد ویژگی ۰٫۱ می‌شود.

</div>

In [ ]:
all_features = np.ones(n_features, dtype=int)

fitness_all = fitness_function(
    all_features,
    X_train,
    y_train,
    classifier,
    cv,
    alpha
)

print("Fitness with all features:", fitness_all)

<div dir="rtl" style="text-align: right">

### تست fitness: انتخاب تصادفی
همان بردار باینری تصادفی سلول ۱۴ (حدود نیمی از ویژگی‌ها) ارزیابی می‌شود. این مقدار نشان می‌دهد یک انتخاب تصادفی چه fitness دارد و الگوریتم باید از آن بهتر شود.

</div>

In [ ]:
fitness_random = fitness_function(
    test_binary,
    X_train,
    y_train,
    classifier,
    cv,
    alpha
)

print("Number of selected features:", test_binary.sum())
print("Fitness with random features:", fitness_random)

<div dir="rtl" style="text-align: right">

### تست fitness: هیچ ویژگی‌ای انتخاب نشده
برای بردار صفر، تابع باید ۱٫۰ برگرداند (جریمه‌ی بیشینه)، چون مدلی بدون ویژگی معنا ندارد.

</div>

In [ ]:
empty_solution = np.zeros(n_features, dtype=int)

fitness_empty = fitness_function(
    empty_solution,
    X_train,
    y_train,
    classifier,
    cv,
    alpha
)

print("Fitness with no selected features:", fitness_empty)

<div dir="rtl" style="text-align: right">

### تست‌های درستی fitness
fitness باید همیشه بین ۰ و ۱ باشد و برای حالت بدون ویژگی دقیقاً ۱٫۰ باشد. اگر شرطی نقض شود برنامه متوقف می‌شود.

</div>

In [ ]:
assert 0 <= fitness_all <= 1
assert 0 <= fitness_random <= 1
assert fitness_empty == 1.0

print("All fitness tests passed.")

<div dir="rtl" style="text-align: right">

### تبدیل کل جمعیت به باینری
همان کار `continuous_to_binary` است ولی یک‌جا روی ماتریس موقعیت‌ها (همه‌ی نهنگ‌ها) انجام می‌شود: سیگموید، سپس مقایسه با اعداد تصادفی. این نسخه داخل `binary_woa` استفاده می‌شود.

</div>

In [ ]:
def population_to_binary(positions, rng):
    probabilities = sigmoid(positions)

    random_values = rng.random(size=positions.shape)

    binary_population = (
        random_values < probabilities
    ).astype(int)

    return binary_population

<div dir="rtl" style="text-align: right">

### آزمایش تبدیل جمعیت
شکل ماتریس باینری، مقادیر یکتای آن (باید فقط ۰ و ۱ باشد) و تعداد ویژگی‌های انتخاب‌شده توسط نهنگ اول چاپ می‌شود.

</div>

In [ ]:
test_population_binary = population_to_binary(
    positions,
    rng
)

print("Binary population shape:", test_population_binary.shape)
print("Unique values:", np.unique(test_population_binary))
print("Selected features in first whale:", test_population_binary[0].sum())

<div dir="rtl" style="text-align: right">

### تابع اصلی `binary_woa`
همه‌ی تکه‌های قبلی در این تابع کنار هم قرار می‌گیرند:
1. **مقداردهی اولیه:** موقعیت‌های تصادفی در $[-6,6]$.
2. **تبدیل به باینری** و 3. **ارزیابی** جمعیت اولیه با fitness.
4. **بهترین اولیه:** نهنگی با کمترین fitness (`argmin`) به‌عنوان بهترین ثبت می‌شود. هم موقعیت پیوسته‌اش (برای حرکت بقیه) و هم بردار باینری‌اش (به‌عنوان جواب) نگه داشته می‌شود.
5. **حلقه‌ی اصلی** (هر تکرار): موقعیت‌ها با `update_whale_positions` به‌روز می‌شوند، به باینری تبدیل می‌شوند، fitness حساب می‌شود و اگر بهترین نهنگ این تکرار از بهترین کلی بهتر بود، جایگزین می‌شود. اگر بهتر نبود بهترین قبلی می‌ماند، پس منحنی همگرایی هرگز بالا نمی‌رود.

نکته‌ی مهم: چون تبدیل به باینری تصادفی است، جواب نهایی همان بردار باینری ذخیره‌شده در لحظه‌ی بهترین fitness است و دوباره از روی موقعیت پیوسته نمونه‌گیری نمی‌شود.

خروجی: بهترین بردار باینری، بهترین fitness و `convergence_curve` (بهترین fitness در شروع و پس از هر تکرار). در هر تکرار پیشرفت چاپ می‌شود.

</div>

In [ ]:
def binary_woa(
    X_train,
    y_train,
    classifier,
    cv,
    population_size=10,
    max_iterations=20,
    alpha=0.9,
    lower_bound=-6,
    upper_bound=6,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    n_features = X_train.shape[1]

    # 1. Initialize continuous whale positions
    positions = rng.uniform(
        lower_bound,
        upper_bound,
        size=(population_size, n_features)
    )

    # 2. Convert initial population to binary
    binary_population = population_to_binary(
        positions,
        rng
    )

    # 3. Evaluate initial population
    fitness_values = np.array([
        fitness_function(
            binary_population[i],
            X_train,
            y_train,
            classifier,
            cv,
            alpha
        )
        for i in range(population_size)
    ])

    # 4. Find initial global best
    best_index = np.argmin(fitness_values)

    best_fitness = fitness_values[best_index]
    best_position = positions[best_index].copy()
    best_binary_solution = binary_population[best_index].copy()

    convergence_curve = [best_fitness]

    # 5. Main WOA iterations
    for iteration in range(max_iterations):

        # Update continuous positions
        positions = update_whale_positions(
            positions=positions,
            best_position=best_position,
            iteration=iteration,
            max_iterations=max_iterations,
            rng=rng,
            lower_bound=lower_bound,
            upper_bound=upper_bound
        )

        # Convert updated positions to binary
        binary_population = population_to_binary(
            positions,
            rng
        )

        # Evaluate new population
        fitness_values = np.array([
            fitness_function(
                binary_population[i],
                X_train,
                y_train,
                classifier,
                cv,
                alpha
            )
            for i in range(population_size)
        ])

        # Find best candidate in current iteration
        current_best_index = np.argmin(fitness_values)
        current_best_fitness = fitness_values[current_best_index]

        # Update global best if improvement is found
        if current_best_fitness < best_fitness:

            best_fitness = current_best_fitness

            best_position = positions[current_best_index].copy()

            best_binary_solution = (
                binary_population[current_best_index].copy()
            )

        # Store best fitness so far
        convergence_curve.append(best_fitness)

        print(
            f"Iteration {iteration + 1}/{max_iterations} "
            f"| Best Fitness: {best_fitness:.6f} "
            f"| Selected Features: {best_binary_solution.sum()}"
        )

    return (
        best_binary_solution,
        best_fitness,
        convergence_curve
    )

<div dir="rtl" style="text-align: right">

### اجرای الگوریتم
`binary_woa` با داده‌ی train، مدل، CV و پارامترها (جمعیت ۱۰، ۲۰ تکرار، $\alpha=0.9$، seed ‏۴۲) اجرا می‌شود. هر خط چاپ‌شده بهترین fitness و تعداد ویژگی‌های بهترین جواب تا آن تکرار است. این اجرا طولانی‌ترین بخش نوت‌بوک است.

</div>

In [ ]:
best_solution, best_fitness, convergence_curve = binary_woa(
    X_train=X_train,
    y_train=y_train,
    classifier=classifier,
    cv=cv,
    population_size=population_size,
    max_iterations=max_iterations,
    alpha=alpha,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    random_state=42
)

<div dir="rtl" style="text-align: right">

### خلاصه‌ی نتیجه
بهترین fitness، شکل بردار جواب، تعداد ویژگی‌های انتخاب‌شده و نشده و مقادیر یکتای بردار (باید فقط ۰ و ۱ باشد) چاپ می‌شود.

</div>

In [ ]:
print("Best Fitness:", best_fitness)

print("Best solution shape:", best_solution.shape)

print("Number of selected features:", best_solution.sum())

print(
    "Number of unselected features:",
    len(best_solution) - best_solution.sum()
)

print("Unique values:", np.unique(best_solution))

<div dir="rtl" style="text-align: right">

### اعتبارسنجی جواب
بررسی می‌کنیم بردار جواب طول درست دارد، فقط ۰ و ۱ دارد و حداقل یک ویژگی انتخاب شده است.

</div>

In [ ]:
assert best_solution.shape == (n_features,)
assert set(np.unique(best_solution)).issubset({0, 1})
assert best_solution.sum() > 0

print("Best solution validation passed.")

<div dir="rtl" style="text-align: right">

### نمودار همگرایی
محور افقی شماره‌ی تکرار (۰ = جمعیت اولیه) و محور عمودی بهترین fitness تا آن تکرار است. نمودار نزولی و پله‌ای است (کاهش فقط وقتی جواب بهتر پیدا شود)، و صاف شدن آن یعنی الگوریتم دیگر بهبودی پیدا نمی‌کند.

</div>

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(len(convergence_curve)),
    convergence_curve,
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("Best Fitness")
plt.title("Binary WOA Convergence Curve")

plt.grid(True)
plt.show()

<div dir="rtl" style="text-align: right">

### مسیر ذخیره‌ی نتایج
`get_results_path` پوشه‌ی `results/<نام دیتاست>/` را می‌سازد و مسیرش را برمی‌گرداند.

</div>

In [ ]:
RESULTS_PATH = get_results_path(PROJECT_PATH)

<div dir="rtl" style="text-align: right">

### نام ویژگی‌های انتخاب‌شده
بردار صفر و یک با ستون‌های `X_train` تطبیق داده می‌شود: ستون‌هایی که مقدارشان ۱ است همان ویژگی‌های انتخاب‌شده‌اند. بیست نام اول و تعداد کل چاپ می‌شود.

</div>

In [ ]:
selected_features = X_train.columns[
    best_solution == 1
].tolist()

print("First 20 selected features:")
print(selected_features[:20])

print("Total selected features:", len(selected_features))

<div dir="rtl" style="text-align: right">

### ذخیره‌ی نتایج
- فهرست ویژگی‌های انتخاب‌شده در `woa_selected_features.csv`
- منحنی همگرایی (fitness در هر تکرار) در `woa_convergence.csv`

این اجرا یک seed دارد و جنبه‌ی آزمایشی دارد؛ اجرای نهایی با سه seed در نوت‌بوک ۰۵ انجام می‌شود.

</div>

In [ ]:
selected_features_df = pd.DataFrame({
    "Feature": selected_features
})

selected_features_df.to_csv(
    RESULTS_PATH / "woa_selected_features.csv",
    index=False
)

convergence_df = pd.DataFrame({
    "Iteration": range(len(convergence_curve)),
    "Best_Fitness": convergence_curve
})

convergence_df.to_csv(
    RESULTS_PATH / "woa_convergence.csv",
    index=False
)

print("Results saved successfully.")